## 1. LCEL

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
# 1. 提示词模版对象
prompt = ChatPromptTemplate.from_template("请用{tone}风格回答：{question}")

# 2. 构建模型实例
model = init_chat_model(model="gpt-3.5-turbo")

# LCEL组合 - 像拼积木一样简单
chain = prompt | model

res = chain.invoke({
    "tone": "幽默",
    "question": "什么是人工智能？"
})

print(res.content)

人工智能就是让机器比人类更聪明，比如说让你的手机比你更了解你自己的生活，比如说让你的电视比你更懂得你想看的节目，当然啦，有时候也可能会让机器比我们这些普通人更懂得如何统治世界，哈哈哈。


In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
# 1. 提示词模版对象
prompt = ChatPromptTemplate.from_template("请用{tone}风格回答：{question}")

# 2. 构建模型实例
model = init_chat_model(model="gpt-3.5-turbo")

# LCEL组合 - 像拼积木一样简单
chain = prompt | model | StrOutputParser()

res = chain.invoke({
    "tone": "幽默",
    "question": "什么是人工智能？"
})

print(res)

人工智能就是让机器比人聪明，让电脑比你还知道你自己想要什么。它就像一个魔法师，只是它不会变出兔子，而是变出了更聪明的计算机程序。所以说，人工智能就是让机器变得越来越聪明，最终可能会超过我们人类，然后欺负我们，不让我们玩电子游戏！


## 2. Runnable组合器

### 2.1  RunnableSequence - 顺序流水线

In [4]:
from langchain_core.runnables import RunnableSequence, RunnableLambda

# 显式创建序列
sequence_chain = RunnableSequence(
    first=RunnableLambda(lambda x: x.upper()),  # 第一步：转换大写
    middle=[RunnableLambda(lambda x: f"HELLO {x} !")],  # 第二步：加装饰
    last=RunnableLambda(lambda x: f"最终：{x}")  # 第三步：加前缀
)

result = sequence_chain.invoke("world")  # 1. RunnableLambda.invoke
print(result)  # 最终：HELLO WORLD !

最终：HELLO WORLD !


### 2.2 RunnableParallel - 并行分叉

In [6]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

# 创建并行任务
parallel_chain = RunnableParallel({
    "length": RunnableLambda(lambda x: len(x)),  # 计算长度
    "uppercase": RunnableLambda(lambda x: x.upper()),  # 转大写
    "reversed": RunnableLambda(lambda x: x[::-1]),  # 反转字符串
    "word_count": RunnableLambda(lambda x: len(x.split()))  # 单词计数
})

result = parallel_chain.invoke("Hello World LangChain")
print(result)
# 输出：
# {
#   'length': 24,
#   'uppercase': 'HELLO WORLD LANGCHAIN',
#   'reversed': 'niahCgnaL dlroW olleH',
#   'word_count': 3
# }

{'length': 21, 'uppercase': 'HELLO WORLD LANGCHAIN', 'reversed': 'niahCgnaL dlroW olleH', 'word_count': 3}


In [11]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import PromptTemplate

prompt =PromptTemplate.from_template(template="{name}")

# 创建并行任务
parallel_chain = prompt | {
    "length": RunnableLambda(lambda x: print(x)),  # 计算长度
    "uppercase": RunnableLambda(lambda x: print(x)),  # 转大写
    "reversed": RunnableLambda(lambda x: x.text[::-1]),  # 反转字符串
    "word_count": RunnableLambda(lambda x: len(x.text.split()))  # 单词计数
}

result = parallel_chain.invoke({"name": "hello world"})
print(result)


text='hello world'
text='hello world'
{'length': None, 'uppercase': None}


In [12]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model(model="gpt-4o-mini", model_provider="openai")

# 两个并行的赏析链
paragraph_1_chain = (
    PromptTemplate.from_template("对这首诗做赏析，分析含义：{poem}")
    | llm | StrOutputParser()
)
paragraph_2_chain = (
    PromptTemplate.from_template("对这首诗做赏析，分析意境：{poem}")
    | llm | StrOutputParser()
)

# 汇总链
summary_chain = (
    PromptTemplate.from_template(
        "第一种赏析：{paragraph_1}\n\n第二种赏析：{paragraph_2}\n\n请比较哪个更好，为什么"
    )
    | llm | StrOutputParser()
)

# 先并行，后汇总
full_chain = {
    "paragraph_1": paragraph_1_chain,
    "paragraph_2": paragraph_2_chain,
} | summary_chain

resp = full_chain.invoke({"poem": "菩提本无树，明镜亦非台，本来无一物，何处惹尘埃。"})
print(resp)

两种赏析都对唐代禅宗大师慧能的诗作进行了深入的分析，但它们的表现和侧重点有所不同。

**第一种赏析的优点**：
1. **结构清晰**：逐句分析的方式使读者容易跟随，理解每一句诗的深意。
2. **哲理性强**：强调了禅宗思想的“无我”“无执”，并将这些哲理与现代生活联系起来，显示出其现实意义。
3. **意境描绘**：对于整体意境的描述比较细腻，能够让读者感受到诗的禅意。

**第二种赏析的优点**：
1. **简洁明了**：语言相对简练，快速传达了诗句的意义，适合快速阅读或简要了解。
2. **哲学思考**：强调了对“空性”和存在的质疑，深化了对生命本质的思考，这种哲学性引发了更深的反思。
3. **意境丰富**：在意境分析部分，通过超越世俗和内心觉悟的探讨，进一步挖掘了诗的内涵。

**比较**：
总体来说，第一种赏析在深度和细致度上表现更为突出，适合希望深入理解诗作的读者。它的语言和结构能够引导读者层层深入，理解禅宗的核心理念。而第二种赏析则更为简洁，适合快速理解或作为初步的引导。 

因此，哪种更好取决于读者的需求：
- 如果希望获得更深入的理解和背景知识，第一种赏析可能更合适。
- 如果需要快速了解诗的核心思想，第二种赏析则更加便捷。

结合以上分析，整体而言，第一种赏析在深度和细致性上更为优秀，而第二种则在简洁和快速理解上占优。


### 2.3 RunnablePassthrough - 输入传递

In [15]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

# 场景1：直接传递输入
# chain = RunnablePassthrough()
# result = chain.invoke({"key": "value"})
# print(result)
chain = RunnableParallel({
    "original":RunnablePassthrough(),
    "uppercase":lambda x: x["text"].upper()
}
)

# chain = RunnableParallel(
#     original=RunnablePassthrough(),            # 保留原始输入
#     uppercase=lambda x: x["text"].upper()      # 添加转换后的字段
# )
result = chain.invoke({"text": "hello"})
print(result)
# {
#  "original": {"text": "hello"}
#  "uppercase":"HELLO"
# }
#
#

{'original': {'text': 'hello'}, 'uppercase': 'HELLO'}


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    "请回答以下问题：{question}\n\n相关背景：{context}"
)

chain = (
    {
        "question": RunnablePassthrough(),           # 用户问题原样传递
        "context": lambda x: retrieve_context(x)     # 检索相关上下文
    }
    | prompt
    | llm
    | StrOutputParser()
)
chain.invoke("xxxx")

### 2.4 RunnableLambda - 自定义逻辑

In [16]:
from langchain_core.runnables import RunnableLambda

def extract_domain(url):
    """从URL中提取域名"""
    return url.split('//')[-1].split('/')[0]

def add_protocol(domain):
    """添加协议前缀"""
    return f"http://{domain}"

# 包装成Runnable
domain_extractor = RunnableLambda(extract_domain)
protocol_adder = RunnableLambda(add_protocol)

# 在链中使用
url_processor = domain_extractor | protocol_adder
result = url_processor.invoke("https://www.example.com/path")
print(result)

http://www.example.com


### 2.5 添加对话历史

In [22]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

# 创建会话存储（以session_id为key）
store = {}
# {"user123":InMemoryChatMessageHistory()--->message:q(HumanMessage)a(AIMessage) }


def get_session_history(session_id: str):
    """
    1. 创建能够存储历史对话的对象InMemoryChatMessageHistory
    2. 通过字典容器隔离不同用户的历史对话
    核心：自己存储到记忆组件的历史对话 自己要取出来，别人不能取出来
    :param session_id:
    :return:
    """
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


# 创建基础链
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是AI助手"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

llm = ChatOpenAI(model="gpt-4o-mini")
chain = prompt | llm

# 包装为带历史记录的链
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# 使用时指定session_id
response_1 = chain_with_history.invoke(
    {"input": "我叫张三"},
    config={"configurable": {"session_id": "123"}}
)
print(response_1.content)

你好，张三！很高兴认识你。有些什么我可以帮助你的呢？


In [24]:
# 后续对话会自动携带历史
response_2 = chain_with_history.invoke(
    {"input": "我叫什么名字？"},
    config={"configurable": {"session_id": "123"}}
)
print(response_2.content)
# AI会回答"你叫张三"，因为历史记录中有这个信息

你叫张三。请问还有其他我可以帮助你的吗？
